In [3]:
!pip install -U scikit-learn
!pip install pandas matplotlib

You should consider upgrading via the 'd:\absa\absa\scripts\python.exe -m pip install --upgrade pip' command.


  Using cached pandas-1.4.1-cp38-cp38-win_amd64.whl (10.6 MB)
  Using cached pytz-2021.3-py2.py3-none-any.whl (503 kB)

You should consider upgrading via the 'd:\absa\absa\scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
import numpy as np
import pandas as pd
import re
import nltk
import spacy
import string
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics import classification_report

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# Stacking
from sklearn.ensemble import StackingClassifier

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download('averaged_perceptron_tagger')
pd.options.mode.chained_assignment = None

# Swap to "data/restaurant_reviews.csv" to run the pipeline on restaurant reviews instead
DATA_PATH = "data/output.csv"

full_df = pd.read_csv(DATA_PATH, nrows=5000)
df = full_df[["Review"]]
df["text"] = df["Review"].astype(str)
full_df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df["text_lower"] = df["text"].str.lower()
df.head()

In [ ]:
# drop the new column created in last cell
#df.drop(["text_lower"], axis=1, inplace=True)

PUNCT_TO_REMOVE = string.punctuation
def remove_punctuation(text):
    """custom function to remove the punctuation"""
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

df["text_wo_punct"] = df["text"].apply(lambda text: remove_punctuation(text))
df.head()

In [ ]:
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))

In [ ]:
STOPWORDS = set(stopwords.words('english'))
def remove_stopwords(text):
    """custom function to remove the stopwords"""
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])

df["text_wo_stop"] = df["text_wo_punct"].apply(lambda text: remove_stopwords(text))
df.head()

In [ ]:
from nltk.stem.porter import PorterStemmer

# Drop the two columns 
#df.drop(["text_wo_stopfreq", "text_wo_stopfreqrare"], axis=1, inplace=True) 

stemmer = PorterStemmer()
def stem_words(text):
    return " ".join([stemmer.stem(word) for word in text.split()])

df["text_stemmed"] = df["text"].apply(lambda text: stem_words(text))
df.head()

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

df["text_lemmatized"] = df["text"].apply(lambda text: lemmatize_words(text))
df.head()

In [ ]:
lemmatizer.lemmatize("running")

In [ ]:
lemmatizer.lemmatize("running", "v") # v for verb

In [ ]:
print("Word is : stripes")
print("Lemma result for verb : ",lemmatizer.lemmatize("stripes", 'v'))
print("Lemma result for noun : ",lemmatizer.lemmatize("stripes", 'n'))

In [ ]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
wordnet_map = {"N":wordnet.NOUN, "V":wordnet.VERB, "J":wordnet.ADJ, "R":wordnet.ADV}
def lemmatize_words(text):
    pos_tagged_text = nltk.pos_tag(text.split())
    return " ".join([lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_tagged_text])

df["text_lemmatized"] = df["text"].apply(lambda text: lemmatize_words(text))
df.head()

In [ ]:
full_df.head()

In [ ]:
df1 = df[full_df['Aspect'] == 'Ambience']
df2 = df[full_df['Aspect'] == 'Food']
df3 = df[full_df['Aspect'] == 'Service']

df1
df2
df3

In [ ]:
grouped = df.groupby(full_df.Aspect)

food = grouped.get_group("Food")
ambience = grouped.get_group("Ambience")
service = grouped.get_group("Service")

In [ ]:
food.to_csv('food.csv')
ambience.to_csv('ambience.csv')
service.to_csv('service.csv')

##Overall 


In [ ]:
corpus = []
for i in range(0, len(full_df)):
  review = re.sub('[^a-zA-Z]', ' ', full_df['Review'][i])
  review = review.lower()
  review = review.split()
  stemmer = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [stemmer.stem(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

In [ ]:
print(corpus[:15])

###Creating a Bag of Words Model

In [ ]:
cv = CountVectorizer(max_features = 151)
X = cv.fit_transform(corpus).toarray()
y = full_df.iloc[:len(corpus), -1].values

###**Splitting the data set into the training set and test se**t

In [ ]:
X_train ,X_test , y_train ,y_test = train_test_split(X , y, test_size = 0.20, random_state = 0)

###Modelling

In [ ]:
models = {} 

nb_models = [RandomForestClassifier, SVC]
for algorithm in nb_models:
    model = algorithm().fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    print(f'---- Algorithm: {algorithm.__name__}\nConfusion Matrix:\n\n {cm} \n\nScore Accuracy: {accuracy_score(y_test,y_pred)}\n Classification Report:\n\n{classification_report(y_test,y_pred)}\n{"-"*25}')
    models[algorithm.__name__] = accuracy_score(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix — {algorithm.__name__}')
    plt.tight_layout()
    plt.show()

##Randomized Search CV

In [ ]:
models = [('RandomForestClassifier',
            RandomForestClassifier(random_state = 0),
           {'max_depth': [100,110],
            'min_samples_leaf': [1, 2, 3],
            'min_samples_split': [5, 6,7],
            'n_estimators': [2300,2400,2500]}),
          
          ('SVC',
           SVC(random_state = 0),
           {'C':[2,2.5,3],
            'gamma' : [0.1, 0.2, 0,3],
            'degree' : [1, 2, 3],
            "kernel":["linear","rbf","sigmoid"],}),
         ]

for name, model, param_grid in models:
    rs = RandomizedSearchCV(estimator = model, 
                            param_distributions = param_grid,
                            scoring='accuracy',
                            cv = 5,
                            n_iter = 10,
                            random_state = 0)
    
    rs.fit(X_train, y_train)
    print(f'{name :20} {rs.best_score_}')
    print(f'{name :20} {rs.best_params_}')

In [ ]:
final_svc = rs.best_estimator_
final_svc

###Evaluation

In [ ]:
# making predictions
final_pred = final_svc.predict(X_test)

In [ ]:
print("Accuracy Score (SVC Final Model):", accuracy_score(y_test, final_pred)*100, "%")
print('Classification Report\n', classification_report(y_test, final_pred))
cm = confusion_matrix(y_test, final_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix — Tuned SVC (Final Model)')
plt.tight_layout()
plt.show()